# Explore the raw retail files

This notebook profiles `data/raw` before any Bronze, Silver, or Gold table is designed.

It does not connect to Postgres, does not read `manifest.json`, and does not write pipeline tables. Each section prints one agreed check. Read the printed tables before locking dedup keys, the business date, or rejection rules.


In [2]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)
pd.set_option("display.max_rows", 40)

ROOT = Path.cwd()
if not (ROOT / "data" / "raw").exists():
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"


def load_raw(raw_dir: Path) -> dict[str, pd.DataFrame]:
    """Load every raw file and flatten event payloads.

    Event files keep business fields inside `payload`. Flattening them here
    lets the later checks name `payload.order_id` without opening each JSON.
    `manifest.json` is metadata and is skipped on purpose.
    """
    frames: dict[str, pd.DataFrame] = {}
    for path in sorted(raw_dir.rglob("*")):
        if not path.is_file() or path.name == "manifest.json":
            continue
        if path.suffix == ".json":
            frame = pd.read_json(path)
        elif path.suffix == ".csv":
            frame = pd.read_csv(path)
        else:
            continue
        if "payload" in frame.columns:
            payload = pd.json_normalize(frame["payload"]).add_prefix("payload.")
            frame = pd.concat([frame.drop(columns=["payload"]), payload], axis=1)
        frames[path.relative_to(raw_dir).as_posix()] = frame
    return frames


frames = load_raw(RAW)
print(f"Loaded {len(frames)} files from {RAW}")
for name, frame in frames.items():
    print(f"  {name}: {len(frame):,} rows, {len(frame.columns)} columns")


Loaded 19 files from /Users/hanii/Documents/HACKTIV8-AIEN/Phase1/milestones/milestone1-omnichannel_retail/data/raw
  events/payment_events.json: 22,582 rows, 7 columns
  events/refund_events.json: 4,121 rows, 7 columns
  events/return_events.json: 3,553 rows, 6 columns
  events/support_events.json: 2,971 rows, 8 columns
  events/web_events.json: 30,986 rows, 9 columns
  inventory/inventory_snapshots.csv: 35,040 rows, 6 columns
  operational/customer_addresses.json: 2,500 rows, 6 columns
  operational/customer_profiles.json: 2,777 rows, 6 columns
  operational/customers.json: 2,500 rows, 5 columns
  operational/order_items.json: 24,960 rows, 7 columns
  operational/order_promotions.json: 15,729 rows, 4 columns
  operational/orders.json: 10,001 rows, 9 columns
  operational/product_categories.json: 86 rows, 6 columns
  operational/products.json: 80 rows, 5 columns
  operational/promotions.json: 12 rows, 7 columns
  operational/sales_channels.json: 4 rows, 3 columns
  operational/stores.j

## 1. Shape

Goal: list every column and its type, so Silver types real fields and does not invent columns.


In [3]:
shape_rows = []
for name, frame in frames.items():
    for column in frame.columns:
        shape_rows.append(
            {
                "file": name,
                "rows": len(frame),
                "column": column,
                "dtype": str(frame[column].dtype),
            }
        )
shape = pd.DataFrame(shape_rows)
shape


,file,rows,column,dtype
0,events/payment_events.json,22582,event_id,str
1,events/payment_events.json,22582,event_type,str
2,events/payment_events.json,22582,ingested_at_utc,str
3,events/payment_events.json,22582,occurred_at_utc,str
4,events/payment_events.json,22582,payload.amount,float64
...,...,...,...,...
108,reference/campaign_spend.csv,855,channel,str
109,reference/campaign_spend.csv,855,spend_amount,float64
110,reference/city_reference.json,8,city_id,str
111,reference/city_reference.json,8,city_name,str


## 2. Nulls and sample rows

Goal: see which fields may be empty, and what a real value looks like, so an empty field is not rejected by accident.

Remember: an empty store id on a digital order is allowed.


In [4]:
null_rows = []
for name, frame in frames.items():
    nulls = frame.isna().sum()
    for column, count in nulls.items():
        if count == 0:
            continue
        null_rows.append(
            {
                "file": name,
                "column": column,
                "nulls": int(count),
                "null_pct": round(count / len(frame), 4),
            }
        )
nulls = pd.DataFrame(null_rows)
if nulls.empty:
    print("No nulls in any loaded column.")
else:
    display(nulls.sort_values(["file", "nulls"], ascending=[True, False]))

for name, frame in frames.items():
    print("\n", name)
    display(frame.head(10))


,file,column,nulls,null_pct
0,events/support_events.json,payload.reason,1305,0.4392
1,events/web_events.json,payload.campaign_id,7785,0.2512
2,operational/orders.json,store_id,7484,0.7483



 events/payment_events.json


,event_id,event_type,ingested_at_utc,occurred_at_utc,payload.amount,payload.order_id,payload.payment_id
0,EVT-00006883,PAYMENT_CAPTURED,2026-07-20T01:14:00Z,2026-07-19T18:14:00Z,2172.62,ORD-001121,PAY-ORD-001121
1,EVT-00010230,PAYMENT_CAPTURED,2026-09-14T03:32:00Z,2026-09-14T02:32:00+07:00,3734.64,ORD-001662,PAY-ORD-001662
2,EVT-00051452,PAYMENT_AUTHORIZED,2026-08-08T08:02:00Z,2026-08-08T02:02:00Z,846.12,ORD-008285,PAY-ORD-008285
3,EVT-00041158,PAYMENT_CAPTURED,2026-08-16T12:44:00Z,2026-08-16T05:44:00Z,1785.92,ORD-006618,PAY-ORD-006618
4,EVT-00055547,PAYMENT_CAPTURED,2026-07-08T22:08:00Z,2026-07-08T17:08:00Z,1057.10,ORD-008937,PAY-ORD-008937
5,EVT-00024618,PAYMENT_AUTHORIZED,2026-07-28T09:55:00Z,2026-07-28T13:55:00+07:00,739.08,ORD-003970,PAY-ORD-003970
6,EVT-00048469,PAYMENT_CAPTURED,2026-08-08T14:39:00Z,2026-08-08T09:39:00Z,1153.45,ORD-007801,PAY-ORD-007801
7,EVT-00003882,PAYMENT_AUTHORIZED,2026-06-30T10:03:00Z,2026-06-30T05:03:00Z,1331.98,ORD-000629,PAY-ORD-000629
8,EVT-00033669,PAYMENT_AUTHORIZED,2026-09-13T17:34:00Z,2026-09-13T11:34:00Z,1697.89,ORD-005425,PAY-ORD-005425
9,EVT-00041855,PAYMENT_AUTHORIZED,2026-08-24T13:01:00Z,2026-08-24T14:01:00+07:00,1277.14,ORD-006728,PAY-ORD-006728



 events/refund_events.json


,event_id,event_type,ingested_at_utc,occurred_at_utc,payload.amount,payload.order_id,payload.refund_id
0,EVT-00040083,REFUND_ISSUED,2026-06-25T20:16:00Z,2026-06-25T16:16:00Z,425.51,ORD-006441,REF-ORD-006441-01
1,EVT-00052826,REFUND_COMPLETED,2026-07-08T16:36:00Z,2026-07-08T11:36:00Z,1623.72,ORD-008503,REF-ORD-008503-01
2,EVT-00010727,REFUND_ISSUED,2026-09-19T07:48:00Z,2026-09-17T01:48:00Z,214.32,ORD-001742,REF-ORD-001742-01
3,EVT-00059354,REFUND_COMPLETED,2026-09-09T17:06:00Z,2026-09-09T09:06:00Z,114.34,ORD-009547,REF-ORD-009547-01
4,EVT-00000252,REFUND_ISSUED,2026-08-15T01:02:00Z,2026-08-14T22:02:00Z,340.56,ORD-000044,REF-ORD-000044-01
5,EVT-00058621,REFUND_ISSUED,2026-08-08T13:58:00Z,2026-08-08T09:58:00Z,1651.40,ORD-009431,REF-ORD-009431-01
6,EVT-00019123,REFUND_COMPLETED,2026-07-15T20:13:00Z,2026-07-15T18:13:00Z,2197.38,ORD-003092,REF-ORD-003092-01
7,EVT-00038878,REFUND_COMPLETED,2026-08-06T20:55:00Z,2026-08-06T17:55:00Z,1312.54,ORD-006250,REF-ORD-006250-01
8,EVT-00012674,REFUND_ISSUED,2026-08-08T13:03:00Z,2026-08-08T09:03:00Z,574.98,ORD-002054,REF-ORD-002054-01
9,EVT-00030956,REFUND_ISSUED,2026-08-31T09:27:00Z,2026-08-31T05:27:00Z,1195.38,ORD-004987,REF-ORD-004987-01


...

,city_id,location_type,source_row_id,store_id,store_name
0,BDG,physical,STORE-ROW-01,STORE-01,Retail Store 01
1,SBY,physical,STORE-ROW-02,STORE-02,Retail Store 02
2,MDN,physical,STORE-ROW-03,STORE-03,Retail Store 03
3,DPS,physical,STORE-ROW-04,STORE-04,Retail Store 04
4,MKS,physical,STORE-ROW-05,STORE-05,Retail Store 05



 reference/campaign_spend.csv


,spend_date,campaign_id,channel,spend_amount
0,2026-06-22,CMP-01,MOBILE_APP,104.82
1,2026-06-22,CMP-02,MARKETPLACE,320.47
2,2026-06-22,CMP-03,WEB,63.73
3,2026-06-22,CMP-04,MOBILE_APP,495.53
4,2026-06-22,CMP-05,MARKETPLACE,110.67
5,2026-06-22,CMP-06,WEB,176.27
6,2026-06-22,CMP-07,MOBILE_APP,368.83
7,2026-06-22,CMP-08,MARKETPLACE,491.41
8,2026-06-22,CMP-09,WEB,410.51
9,2026-06-22,CMP-10,MOBILE_APP,277.34



 reference/city_reference.json


,city_id,city_name,country
0,JKT,Synthetic City JKT,ID
1,BDG,Synthetic City BDG,ID
2,SBY,Synthetic City SBY,ID
3,MDN,Synthetic City MDN,ID
4,DPS,Synthetic City DPS,ID
5,MKS,Synthetic City MKS,ID
6,BPN,Synthetic City BPN,ID
7,PLM,Synthetic City PLM,ID


In [15]:
frames["operational/orders.json"]["status"].value_counts()

status
PLACED       3493
FULFILLED    2561
CONFIRMED    1967
CANCELLED    1215
RETURNED      765
Name: count, dtype: int64

In [18]:
cancelled_orders = frames["operational/orders.json"][frames["operational/orders.json"]["status"] == "CANCELLED"]
display(cancelled_orders)

,customer_id,order_id,ordered_at_utc,sales_channel,shipping_revenue,source_row_id,status,store_id,updated_at_utc
5,CUST-00630,ORD-000006,2026-08-08T18:59:00Z,WEB,0.0,ORDER-ROW-0000006,CANCELLED,NaN,2026-08-09T00:59:00Z
16,CUST-00602,ORD-000017,2026-08-01T11:00:00Z,MARKETPLACE,7.5,ORDER-ROW-0000017,CANCELLED,NaN,2026-08-02T16:00:00Z
21,CUST-01333,ORD-000022,2026-08-18T21:28:00Z,MOBILE_APP,0.0,ORDER-ROW-0000022,CANCELLED,NaN,2026-08-20T00:28:00Z
22,CUST-02486,ORD-000023,2026-07-27T06:15:00Z,STORE,3.5,ORDER-ROW-0000023,CANCELLED,STORE-01,2026-07-28T08:15:00Z
29,CUST-01324,ORD-000030,2026-08-15T20:43:00Z,WEB,7.5,ORDER-ROW-0000030,CANCELLED,NaN,2026-08-16T23:43:00Z
...,...,...,...,...,...,...,...,...,...
9946,CUST-00212,ORD-009947,2026-07-17T16:40:00Z,MOBILE_APP,12.0,ORDER-ROW-0009947,CANCELLED,NaN,2026-07-18T01:40:00Z
9947,CUST-02361,ORD-009948,2026-07-13T13:49:00Z,MARKETPLACE,12.0,ORDER-ROW-0009948,CANCELLED,NaN,2026-07-14T07:49:00Z
9952,CUST-00972,ORD-009953,2026-09-04T21:41:00Z,STORE,3.5,ORDER-ROW-0009953,CANCELLED,STORE-02,2026-09-05T17:41:00Z
9981,CUST-01888,ORD-009982,2026-07-09T17:44:00Z,WEB,0.0,ORDER-ROW-0009982,CANCELLED,NaN,2026-07-10T14:44:00Z


## 3. Candidate keys and duplicates

Goal: see which id repeats, so Silver keeps one winner and Bronze keeps every copy.

Remember: a repeated event id keeps one Silver winner. A repeated order id means many payments, and those rows stay.


In [5]:
KEY_CANDIDATES = {
    "events/payment_events.json": ["event_id", "payload.payment_id", "payload.order_id"],
    "events/refund_events.json": ["event_id", "payload.refund_id", "payload.order_id"],
    "events/return_events.json": ["event_id", "payload.return_id", "payload.order_id"],
    "events/support_events.json": ["event_id", "payload.ticket_id", "payload.order_id"],
    "events/web_events.json": ["event_id", "payload.session_id", "payload.order_id"],
    "inventory/inventory_snapshots.csv": [["product_id", "location_id", "snapshot_date"], "product_id"],
    "operational/customer_addresses.json": ["address_id", "source_row_id", "customer_id"],
    "operational/customer_profiles.json": ["source_row_id", "customer_id"],
    "operational/customers.json": ["customer_id", "source_row_id"],
    "operational/order_items.json": ["order_item_id", "source_row_id", "order_id"],
    "operational/order_promotions.json": ["source_row_id", ["order_id", "promotion_id"], "order_id"],
    "operational/orders.json": ["order_id", "source_row_id"],
    "operational/product_categories.json": ["source_row_id", "product_id"],
    "operational/products.json": ["product_id", "source_row_id", "sku"],
    "operational/promotions.json": ["promotion_id", "source_row_id"],
    "operational/sales_channels.json": ["channel_id", "source_row_id"],
    "operational/stores.json": ["store_id", "source_row_id"],
    "reference/campaign_spend.csv": [["spend_date", "campaign_id", "channel"], "campaign_id"],
    "reference/city_reference.json": ["city_id"],
}


def duplicate_report(frame: pd.DataFrame, keys) -> dict:
    """Count rows that share a candidate key, including composite keys."""
    missing = [key for key in keys if key not in frame.columns] if isinstance(keys, list) else []
    if isinstance(keys, list) and any(isinstance(key, str) and key not in frame.columns for key in keys):
        # Composite keys are lists; a missing part makes the key unusable.
        if not all(key in frame.columns for key in keys):
            return {"rows": len(frame), "unique_keys": None, "duplicate_rows": None, "status": "column missing"}
    if isinstance(keys, str):
        keys = [keys]
    if not all(key in frame.columns for key in keys):
        return {"rows": len(frame), "unique_keys": None, "duplicate_rows": None, "status": "column missing"}
    unique_keys = frame.groupby(keys, dropna=False).ngroups
    duplicate_rows = int(frame.duplicated(keys, keep=False).sum())
    return {
        "rows": len(frame),
        "unique_keys": int(unique_keys),
        "duplicate_rows": duplicate_rows,
        "status": "duplicates" if duplicate_rows else "unique",
    }


dup_rows = []
for name, candidates in KEY_CANDIDATES.items():
    frame = frames[name]
    for keys in candidates:
        label = " + ".join(keys) if isinstance(keys, list) else keys
        report = duplicate_report(frame, keys if isinstance(keys, list) else [keys])
        dup_rows.append({"file": name, "key": label, **report})
pd.DataFrame(dup_rows)


,file,key,rows,unique_keys,duplicate_rows,status
0,events/payment_events.json,event_id,22582,21915,1334,duplicates
1,events/payment_events.json,payload.payment_id,22582,10000,22582,duplicates
2,events/payment_events.json,payload.order_id,22582,10000,22582,duplicates
3,events/refund_events.json,event_id,4121,3982,278,duplicates
4,events/refund_events.json,payload.refund_id,4121,1991,4121,duplicates
...,...,...,...,...,...,...
41,operational/stores.json,store_id,5,5,0,unique
42,operational/stores.json,source_row_id,5,5,0,unique
43,reference/campaign_spend.csv,spend_date + campaign_id + channel,855,855,0,unique
44,reference/campaign_spend.csv,campaign_id,855,10,855,duplicates


In [6]:
def rows_that_differ(frame, keys):
    key_cols = keys if isinstance(keys, list) else [keys]
    if not all(column in frame.columns for column in key_cols):
        return None
    compare_cols = [column for column in frame.columns if column not in key_cols]
    repeated = frame[frame.duplicated(key_cols, keep=False)]
    if repeated.empty:
        return repeated
    differs = (
        repeated.groupby(key_cols, dropna=False)[compare_cols]
        .nunique(dropna=False)
        .gt(1)
        .any(axis=1)
    )
    changed_keys = differs[differs].index
    indexed = repeated.set_index(key_cols)
    return indexed.loc[indexed.index.isin(changed_keys)].reset_index().sort_values(key_cols)

summary = []
details = {}
for name, candidates in KEY_CANDIDATES.items():
    keys = candidates[0]
    changed = rows_that_differ(frames[name], keys)
    label = " + ".join(keys) if isinstance(keys, list) else keys
    summary.append({
        "file": name,
        "key": label,
        "rows": 0 if changed is None else len(changed),
    })
    if changed is not None and len(changed):
        details[name] = changed

pd.DataFrame(summary).sort_values("rows", ascending=False)

,file,key,rows
11,operational/orders.json,order_id,2
0,events/payment_events.json,event_id,0
10,operational/order_promotions.json,source_row_id,0
17,reference/campaign_spend.csv,spend_date + campaign_id + channel,0
16,operational/stores.json,store_id,0
15,operational/sales_channels.json,channel_id,0
14,operational/promotions.json,promotion_id,0
13,operational/products.json,product_id,0
12,operational/product_categories.json,source_row_id,0
9,operational/order_items.json,order_item_id,0


In [7]:
# details["operational/orders.json"].sort_values("updated_at_utc")

view = details["operational/orders.json"].copy()
view.insert(0, "source_line_number", view.index + 1)
view.sort_values(["updated_at_utc", "source_line_number"])

,source_line_number,order_id,customer_id,ordered_at_utc,sales_channel,shipping_revenue,source_row_id,status,store_id,updated_at_utc
0,1,ORD-000001,CUST-01757,2026-08-20T06:16:00Z,MOBILE_APP,12.0,ORDER-ROW-0000001,PLACED,NaN,2026-08-20T23:16:00Z
1,2,ORD-000001,CUST-01757,2026-08-20T06:16:00Z,MOBILE_APP,12.0,ORDER-ROW-DUPLICATE-0001,PLACED,NaN,2026-08-20T23:16:00Z


In [8]:
payments = frames["events/payment_events.json"]
counts = payments["event_id"].value_counts()
repeated_ids = counts[counts > 1].index

(
    payments[payments["event_id"].isin(repeated_ids)]
    .sort_values(["event_id", "occurred_at_utc"])
    [["event_id", "event_type", "occurred_at_utc", "payload.order_id", "payload.payment_id", "payload.amount"]]
)

,event_id,event_type,occurred_at_utc,payload.order_id,payload.payment_id,payload.amount
18260,EVT-00000062,PAYMENT_CAPTURED,2026-08-21T04:31:00Z,ORD-000012,PAY-ORD-000012,1711.65
20349,EVT-00000062,PAYMENT_CAPTURED,2026-08-21T04:31:00Z,ORD-000012,PAY-ORD-000012,1711.65
2007,EVT-00000155,PAYMENT_CAPTURED,2026-07-27T19:32:00Z,ORD-000029,PAY-ORD-000029,616.20
21052,EVT-00000155,PAYMENT_CAPTURED,2026-07-27T19:32:00Z,ORD-000029,PAY-ORD-000029,616.20
11622,EVT-00000217,PAYMENT_CAPTURED,2026-07-14T15:14:00Z,ORD-000039,PAY-ORD-000039,282.78
...,...,...,...,...,...,...
16597,EVT-00062062,PAYMENT_AUTHORIZED,2026-06-27T04:00:00+07:00,ORD-009977,PAY-ORD-009977,1908.13
9444,EVT-00062093,PAYMENT_CAPTURED,2026-06-26T23:30:00Z,ORD-009981,PAY-ORD-009981,886.22
20631,EVT-00062093,PAYMENT_CAPTURED,2026-06-26T23:30:00Z,ORD-009981,PAY-ORD-009981,886.22
5658,EVT-00062186,PAYMENT_AUTHORIZED,2026-09-03T02:59:00Z,ORD-009997,PAY-ORD-009997,1727.40


## 4. Timestamps and timezones

Goal: see whether a timezone is present, so Silver converts clock times to UTC.

Remember: parse each clock time with its own `Z` or offset, convert to UTC, and use the UTC calendar day. Date-only fields stay calendar dates.


In [9]:
def timestamp_columns(frame: pd.DataFrame) -> list[str]:
    names = []
    for column in frame.columns:
        lowered = column.lower()
        if any(token in lowered for token in ("_at", "date", "valid_from", "valid_to")):
            names.append(column)
    return names


def timezone_shape(value) -> str:
    """Classify the raw string, before parsing hides the offset."""
    if pd.isna(value):
        return "null"
    text = str(value)
    if text.endswith("Z"):
        return "utc_z"
    if "+" in text[10:] or text[10:].find("-") > 0:
        return "offset"
    return "no_timezone"


tz_rows = []
for name, frame in frames.items():
    for column in timestamp_columns(frame):
        raw = frame[column]
        parsed = pd.to_datetime(raw, utc=True, errors="coerce")
        shapes = raw.map(timezone_shape).value_counts()
        tz_rows.append(
            {
                "file": name,
                "column": column,
                "nulls": int(raw.isna().sum()),
                "parse_failures": int(parsed.isna().sum() - raw.isna().sum()),
                "min": parsed.min(),
                "max": parsed.max(),
                "shapes": shapes.to_dict(),
                "sample": None if raw.dropna().empty else str(raw.dropna().iloc[0]),
            }
        )
pd.DataFrame(tz_rows)


,file,column,nulls,parse_failures,min,max,shapes,sample
0,events/payment_events.json,ingested_at_utc,0,0,2026-06-22 01:57:00+00:00,2026-09-22 05:18:00+00:00,{'utc_z': 22582},2026-07-20T01:14:00Z
1,events/payment_events.json,occurred_at_utc,0,0,2026-06-22 00:16:00+00:00,2026-09-20 01:33:00+00:00,"{'utc_z': 20581, 'offset': 2001}",2026-07-19T18:14:00Z
2,events/refund_events.json,ingested_at_utc,0,0,2026-06-25 01:11:00+00:00,2026-09-25 15:52:00+00:00,{'utc_z': 4121},2026-06-25T20:16:00Z
3,events/refund_events.json,occurred_at_utc,0,0,2026-06-25 00:11:00+00:00,2026-09-23 20:01:00+00:00,"{'utc_z': 3761, 'offset': 360}",2026-06-25T16:16:00Z
4,events/return_events.json,ingested_at_utc,0,0,2026-06-27 02:11:00+00:00,2026-10-01 17:09:00+00:00,{'utc_z': 3553},2026-08-22T09:10:00Z
5,events/return_events.json,occurred_at_utc,0,0,2026-06-27 00:11:00+00:00,2026-09-29 19:47:00+00:00,"{'utc_z': 3228, 'offset': 325}",2026-08-22T01:10:00Z
6,events/support_events.json,ingested_at_utc,0,0,2026-06-24 09:59:00+00:00,2026-10-10 00:27:00+00:00,{'utc_z': 2971},2026-09-05T23:50:00Z
7,events/support_events.json,occurred_at_utc,0,0,2026-06-24 06:59:00+00:00,2026-10-09 20:27:00+00:00,"{'utc_z': 2703, 'offset': 268}",2026-09-05T20:50:00Z
8,events/web_events.json,ingested_at_utc,0,0,2026-06-22 01:55:00+00:00,2026-09-22 03:55:00+00:00,{'utc_z': 30986},2026-07-11T10:00:00Z
9,events/web_events.json,occurred_at_utc,0,0,2026-06-21 22:11:00+00:00,2026-09-19 23:46:00+00:00,"{'utc_z': 28103, 'offset': 2883}",2026-07-11T07:00:00Z


## 5. Negative numbers

Goal: see which numbers are below zero, so Silver rejects them with a reason.

Remember: a negative quantity rejects that item row only. Bronze keeps the original row. The rest of the order stays.


In [10]:
negative_rows = []
for name, frame in frames.items():
    numeric = frame.select_dtypes(include="number")
    for column in numeric.columns:
        series = numeric[column]
        negatives = series < 0
        if not negatives.any():
            continue
        negative_rows.append(
            {
                "file": name,
                "column": column,
                "negative_rows": int(negatives.sum()),
                "min": series.min(),
            }
        )
if not negative_rows:
    print("No negative numeric values.")
pd.DataFrame(negative_rows)


,file,column,negative_rows,min
0,operational/order_items.json,quantity,1,-2


## 6. Split payments and partial refunds

Goal: see how many money events belong to one order, so Gold sums them to one amount per order before any join.

Remember: sum `PAYMENT_CAPTURED` into captured amount. Sum `REFUND_COMPLETED` once per refund. Leave authorized, failed, and issued events out of those totals.


In [12]:
payments = frames["events/payment_events.json"]
refunds = frames["events/refund_events.json"]
print("Payment event types")
display(payments["event_type"].value_counts())
print("Refund event types")
display(refunds["event_type"].value_counts())

captured = payments[payments["event_type"].astype(str).str.contains("CAPTURE", case=False)]
captures_per_order = captured.groupby("payload.order_id").size()
print(f"Orders with captured payments: {captures_per_order.size:,}")
print(f"Orders with more than one captured payment: {int((captures_per_order > 1).sum()):,}")
display(captures_per_order.value_counts().sort_index().rename("orders").to_frame())

captured_amount = captured.groupby("payload.order_id")["payload.amount"].sum().rename("captured_amount")
refund_amount = refunds.groupby("payload.order_id")["payload.amount"].sum().rename("refund_amount")
money = pd.concat([captured_amount, refund_amount], axis=1).fillna(0)
partial = money[(money["refund_amount"] > 0) & (money["refund_amount"] < money["captured_amount"])]
full = money[(money["refund_amount"] > 0) & (money["refund_amount"] == money["captured_amount"])]
over = money[money["refund_amount"] > money["captured_amount"]]
print(f"Partial refunds: {len(partial):,}")
print(f"Refund equals captured: {len(full):,}")
print(f"Refund exceeds captured: {len(over):,}")
display(partial.head(5))


Payment event types


event_type
PAYMENT_CAPTURED      11030
PAYMENT_AUTHORIZED    10302
PAYMENT_FAILED         1250
Name: count, dtype: int64

Refund event types


event_type
REFUND_COMPLETED    2064
REFUND_ISSUED       2057
Name: count, dtype: int64

Orders with captured payments: 8,785
Orders with more than one captured payment: 2,115


,orders
1,6670
2,1985
3,130


Partial refunds: 920
Refund equals captured: 224
Refund exceeds captured: 847


,captured_amount,refund_amount
payload.order_id,,
ORD-000035,1236.25,865.38
ORD-000038,1376.06,550.42
ORD-000039,565.56,282.78
ORD-000052,3601.87,2521.30
ORD-000053,1442.94,577.18


## 7. Late events and out-of-order timestamps

Goal: see events that arrive out of time order, so Silver keeps them and uses the time the business event happened.

Remember: keep every event. Business time is `occurred_at` converted to UTC.


In [8]:
orders = frames["operational/orders.json"].copy()
orders["ordered_at_utc"] = pd.to_datetime(orders["ordered_at_utc"], utc=True, errors="coerce")
# order_id is not unique in the raw file, so map each event to the earliest copy.
order_time = orders.groupby("order_id")["ordered_at_utc"].min()

late_rows = []
for name, frame in frames.items():
    if "occurred_at_utc" not in frame.columns or "ingested_at_utc" not in frame.columns:
        continue
    occurred = pd.to_datetime(frame["occurred_at_utc"], utc=True, errors="coerce")
    ingested = pd.to_datetime(frame["ingested_at_utc"], utc=True, errors="coerce")
    lag_hours = (ingested - occurred).dt.total_seconds() / 3600
    before_order = 0
    if "payload.order_id" in frame.columns:
        event_orders = frame["payload.order_id"].map(order_time)
        before_order = int((occurred < event_orders).sum())
    late_rows.append(
        {
            "file": name,
            "rows": len(frame),
            "late_ingested_after_occurred": int((ingested > occurred).sum()),
            "ingested_before_occurred": int((ingested < occurred).sum()),
            "same_timestamp": int((ingested == occurred).sum()),
            "lag_hours_p50": round(lag_hours.median(), 2),
            "lag_hours_p95": round(lag_hours.quantile(0.95), 2),
            "occurred_before_order": before_order,
        }
    )
pd.DataFrame(late_rows)


,file,rows,late_ingested_after_occurred,ingested_before_occurred,same_timestamp,lag_hours_p50,lag_hours_p95,occurred_before_order
0,events/payment_events.json,22582,22582,0,0,5.0,50.0,0
1,events/refund_events.json,4121,4121,0,0,5.0,50.0,0
2,events/return_events.json,3553,3553,0,0,5.0,49.0,0
3,events/support_events.json,2971,2971,0,0,5.0,49.5,0
4,events/web_events.json,30986,30986,0,0,5.0,50.0,20653


## 8. Missing references

Goal: see ids whose parent row is absent, so Silver can reject or flag that row with a reason.

Remember: an empty store id on a digital order is allowed. A missing product rejects that item row only. The rest of the order stays.


In [9]:
def id_set(frame: pd.DataFrame, column: str) -> set[str]:
    return set(frame[column].dropna().astype(str))


def missing_refs(frame: pd.DataFrame, column: str, parents: set[str]) -> dict:
    series = frame[column]
    present = series.dropna().astype(str)
    missing = ~present.isin(parents)
    return {
        "rows": len(frame),
        "nulls": int(series.isna().sum()),
        "missing_rows": int(missing.sum()),
        "missing_ids": int(present[missing].nunique()),
    }


customers = id_set(frames["operational/customers.json"], "customer_id")
products = id_set(frames["operational/products.json"], "product_id")
order_ids = id_set(frames["operational/orders.json"], "order_id")
stores = id_set(frames["operational/stores.json"], "store_id")
promotions = id_set(frames["operational/promotions.json"], "promotion_id")
channels = id_set(frames["operational/sales_channels.json"], "channel_id")
cities = id_set(frames["reference/city_reference.json"], "city_id")

checks = [
    ("operational/orders.json", "customer_id", customers),
    ("operational/orders.json", "store_id", stores),
    ("operational/orders.json", "sales_channel", channels),
    ("operational/order_items.json", "order_id", order_ids),
    ("operational/order_items.json", "product_id", products),
    ("operational/order_promotions.json", "order_id", order_ids),
    ("operational/order_promotions.json", "promotion_id", promotions),
    ("operational/customer_profiles.json", "customer_id", customers),
    ("operational/customer_addresses.json", "customer_id", customers),
    ("operational/customer_addresses.json", "city_id", cities),
    ("operational/product_categories.json", "product_id", products),
    ("inventory/inventory_snapshots.csv", "product_id", products),
    ("events/payment_events.json", "payload.order_id", order_ids),
    ("events/refund_events.json", "payload.order_id", order_ids),
    ("events/return_events.json", "payload.order_id", order_ids),
    ("events/support_events.json", "payload.customer_id", customers),
    ("events/support_events.json", "payload.order_id", order_ids),
    ("events/web_events.json", "payload.customer_id", customers),
    ("events/web_events.json", "payload.order_id", order_ids),
    ("reference/campaign_spend.csv", "channel", channels),
]

ref_rows = []
for name, column, parents in checks:
    report = missing_refs(frames[name], column, parents)
    ref_rows.append({"file": name, "column": column, **report})
pd.DataFrame(ref_rows)


,file,column,rows,nulls,missing_rows,missing_ids
0,operational/orders.json,customer_id,10001,0,0,0
1,operational/orders.json,store_id,10001,7484,0,0
2,operational/orders.json,sales_channel,10001,0,0,0
3,operational/order_items.json,order_id,24960,0,0,0
4,operational/order_items.json,product_id,24960,0,3,1
5,operational/order_promotions.json,order_id,15729,0,0,0
6,operational/order_promotions.json,promotion_id,15729,0,0,0
7,operational/customer_profiles.json,customer_id,2777,0,0,0
8,operational/customer_addresses.json,customer_id,2500,0,0,0
9,operational/customer_addresses.json,city_id,2500,0,0,0


## 9. Profile and category changes over time

Goal: see when a customer or product has more than one version, so Silver picks the version valid at the order time.

Remember: use the version whose `valid_from` and `valid_to` contain the order time. If there is only one version, use that one.


In [10]:
def version_report(frame: pd.DataFrame, entity: str) -> dict:
    """Count versions and overlapping validity windows for one entity."""
    work = frame.copy()
    work["valid_from_utc"] = pd.to_datetime(work["valid_from_utc"], utc=True, errors="coerce")
    work["valid_to_utc"] = pd.to_datetime(work["valid_to_utc"], utc=True, errors="coerce")
    work = work.sort_values([entity, "valid_from_utc"])
    versions = work.groupby(entity).size()
    previous_to = work.groupby(entity)["valid_to_utc"].shift(1)
    has_previous = work.groupby(entity).cumcount() > 0
    previous_open = has_previous & previous_to.isna()
    overlaps_closed = has_previous & previous_to.notna() & (work["valid_from_utc"] < previous_to)
    return {
        "entities": int(versions.size),
        "entities_with_many_versions": int((versions > 1).sum()),
        "max_versions": int(versions.max()),
        "open_ended_versions": int(work["valid_to_utc"].isna().sum()),
        "overlapping_rows": int((previous_open | overlaps_closed).sum()),
    }


pd.DataFrame(
    [
        {"file": "operational/customer_profiles.json", **version_report(frames["operational/customer_profiles.json"], "customer_id")},
        {"file": "operational/customer_addresses.json", **version_report(frames["operational/customer_addresses.json"], "customer_id")},
        {"file": "operational/product_categories.json", **version_report(frames["operational/product_categories.json"], "product_id")},
    ]
)


,file,entities,entities_with_many_versions,max_versions,open_ended_versions,overlapping_rows
0,operational/customer_profiles.json,2500,277,2,0,0
1,operational/customer_addresses.json,2500,0,1,0,0
2,operational/product_categories.json,80,6,2,0,0


## 10. Cancelled orders with captured payment

Goal: see orders marked cancelled after money was taken, so Gold keeps the captured amount.

Remember: keep both facts when this case appears. Do not drop the captured amount because the status says cancelled.


In [11]:
orders = frames["operational/orders.json"]
print("Order statuses")
display(orders["status"].value_counts(dropna=False))

cancelled = orders[orders["status"].astype(str).str.contains("CANCEL", case=False)]
captured_orders = set(
    payments.loc[
        payments["event_type"].astype(str).str.contains("CAPTURE", case=False),
        "payload.order_id",
    ].dropna().astype(str)
)
cancelled_captured = cancelled[cancelled["order_id"].astype(str).isin(captured_orders)]
print(f"Cancelled orders: {len(cancelled):,}")
print(f"Cancelled orders with a captured payment: {len(cancelled_captured):,}")
display(cancelled_captured[["order_id", "customer_id", "status", "ordered_at_utc"]].head(5))


Order statuses


status
PLACED       3493
FULFILLED    2561
CONFIRMED    1967
CANCELLED    1215
RETURNED      765
Name: count, dtype: int64

Cancelled orders: 1,215
Cancelled orders with a captured payment: 0


,order_id,customer_id,status,ordered_at_utc


## 11. Missing inventory snapshot days

Goal: see product and location days with no snapshot, so a missing day is not treated as zero stock.

Remember: a missing day stays unknown. Do not insert quantity 0. Stockout is decided only on days that have a snapshot.


In [12]:
inventory = frames["inventory/inventory_snapshots.csv"].copy()
inventory["snapshot_date"] = pd.to_datetime(inventory["snapshot_date"], errors="coerce")
span_start = inventory["snapshot_date"].min()
span_end = inventory["snapshot_date"].max()
expected_days = pd.date_range(span_start, span_end, freq="D")
print(f"Snapshot span: {span_start.date()} to {span_end.date()} ({len(expected_days)} days)")

observed_days = set(inventory["snapshot_date"].dropna().dt.normalize().unique())
missing_global = [day for day in expected_days if day not in observed_days]
print(f"Calendar days with no snapshot anywhere: {len(missing_global)}")

gap_rows = []
for keys, group in inventory.groupby(["product_id", "location_id"], dropna=False):
    days = set(group["snapshot_date"].dropna().dt.normalize())
    local_start, local_end = min(days), max(days)
    local_expected = pd.date_range(local_start, local_end, freq="D")
    missing = len([day for day in local_expected if day not in days])
    if missing:
        gap_rows.append(
            {
                "product_id": keys[0],
                "location_id": keys[1],
                "observed_days": len(days),
                "missing_days_inside_span": missing,
            }
        )
gaps = pd.DataFrame(gap_rows)
print(f"Product-location series with an internal gap: {len(gaps):,}")
if not gaps.empty:
    display(gaps.sort_values("missing_days_inside_span", ascending=False).head(10))


Snapshot span: 2026-06-22 to 2026-09-19 (90 days)
Calendar days with no snapshot anywhere: 0
Product-location series with an internal gap: 400


,product_id,location_id,observed_days,missing_days_inside_span
117,PROD-00024,STORE-03,87,3
115,PROD-00024,STORE-01,87,3
286,PROD-00058,STORE-02,87,3
118,PROD-00024,STORE-04,87,3
119,PROD-00024,STORE-05,87,3
120,PROD-00025,STORE-01,87,3
121,PROD-00025,STORE-02,87,3
122,PROD-00025,STORE-03,87,3
123,PROD-00025,STORE-04,87,3
124,PROD-00025,STORE-05,87,3


## 12. Promotions and campaigns that can repeat a fact

Goal: see orders tied to more than one promotion or campaign, so Gold counts revenue once before attaching them.

Remember: count each order's revenue once and store the promotion count on that row.


In [13]:
promos = frames["operational/order_promotions.json"]
promos_per_order = promos.groupby("order_id").size()
print(f"Orders with a promotion: {promos_per_order.size:,}")
print(f"Orders with more than one promotion: {int((promos_per_order > 1).sum()):,}")
display(promos_per_order.value_counts().sort_index().rename("orders").to_frame())

web = frames["events/web_events.json"]
attributed = web.dropna(subset=["payload.order_id", "payload.campaign_id"])
campaigns_per_order = attributed.groupby("payload.order_id")["payload.campaign_id"].nunique()
print(f"Orders with a web campaign id: {campaigns_per_order.size:,}")
print(f"Orders with more than one campaign id: {int((campaigns_per_order > 1).sum()):,}")
display(campaigns_per_order.value_counts().sort_index().rename("orders").to_frame())

spend = frames["reference/campaign_spend.csv"]
spend_grain = ["spend_date", "campaign_id", "channel"]
spend_dupes = int(spend.duplicated(spend_grain, keep=False).sum())
print(f"Campaign spend rows: {len(spend):,}")
print(f"Campaign spend rows sharing the same day, campaign, and channel: {spend_dupes:,}")


Orders with a promotion: 9,064
Orders with more than one promotion: 6,665


,orders
1,2399
2,6665


Orders with a web campaign id: 7,483
Orders with more than one campaign id: 0


,orders
payload.campaign_id,
1,7483


Campaign spend rows: 855
Campaign spend rows sharing the same day, campaign, and channel: 0
